# TrendMiners: audit and K-Means comparison

Run the cells in order in Google Colab. This notebook prepares a **new exploratory implementation**, not a reproduction of the missing original notebook. It compares ordinary Logistic Regression with TF-IDF plus K-Means distance features. No improvement is assumed.

The source archive and translation method still need confirmation. The old test results have already been inspected. Use a fresh holdout for stronger final evidence.

In [ ]:
%pip install -q "numpy>=1.26,<3" "scipy>=1.11,<2" "scikit-learn>=1.4,<2"

In [ ]:
from pathlib import Path
import json
source_files = json.loads("{\"audit_project.py\":\"\\\"\\\"\\\"Audit the supplied archive without loading executable pickle/joblib artifacts.\\\"\\\"\\\"\\nimport argparse\\nimport ast\\nimport csv\\nimport hashlib\\nimport io\\nimport json\\nimport struct\\nimport unicodedata\\nimport zipfile\\nfrom collections import Counter, defaultdict\\nfrom pathlib import Path\\n\\n\\ndef normalize(text):\\n    return ' '.join(unicodedata.normalize('NFKC', text or '').casefold().split())\\n\\n\\ndef text_key(text):\\n    text = normalize(text)\\n    return hashlib.sha256(text.encode('utf-8')).hexdigest() if text else None\\n\\n\\ndef numeric_npy(payload):\\n    stream = io.BytesIO(payload)\\n    if stream.read(6) != b'\\\\x93NUMPY':\\n        raise ValueError('Invalid NPY header')\\n    version = stream.read(2)[0]\\n    size = 2 if version == 1 else 4\\n    length = int.from_bytes(stream.read(size), 'little')\\n    header = ast.literal_eval(stream.read(length).decode('latin1'))\\n    codes = {'<i8': 'q', '<i4': 'i', '<f8': 'd', '<f4': 'f'}\\n    if header['descr'] not in codes or header['fortran_order']:\\n        raise ValueError('Unsupported numeric array')\\n    values = [v[0] for v in struct.iter_unpack('<' + codes[header['descr']], stream.read())]\\n    return values\\n\\n\\ndef metrics(matrix):\\n    total = sum(map(sum, matrix))\\n    classes = []\\n    for i, row in enumerate(matrix):\\n        support, predicted = sum(row), sum(r[i] for r in matrix)\\n        p = row[i] / predicted if predicted else 0\\n        r = row[i] / support if support else 0\\n        classes.append({'precision': p, 'recall': r, 'f1': 2*p*r/(p+r) if p+r else 0, 'support': support})\\n    return {'accuracy': sum(matrix[i][i] for i in range(3))/total,\\n            'macro_f1': sum(c['f1'] for c in classes)/3, 'per_class': classes, 'matrix': matrix}\\n\\n\\ndef audit(archive_path):\\n    archive_path = Path(archive_path)\\n    digest = hashlib.file_digest(archive_path.open('rb'), 'sha256').hexdigest()\\n    result = {'archive': archive_path.name, 'archive_sha256': digest,\\n              'normalization': 'Unicode NFKC, casefold, whitespace collapse; hashes only in memory',\\n              'code_files': [], 'splits': {}}\\n    keys = {}\\n    test_truth = []\\n    with zipfile.ZipFile(archive_path) as archive:\\n        names = archive.namelist()\\n        result['code_files'] = [n for n in names if n.endswith(('.py', '.ipynb'))]\\n        for split in ['train', 'test']:\\n            name = next(n for n in names if n.endswith('/data/' + split + '_clean_en.csv'))\\n            counters = {'review_body': Counter(), 'review_clean_en': Counter()}\\n            labelsets = {k: defaultdict(set) for k in counters}\\n            blanks = Counter(); rating_counts = Counter(); mismatches = 0; n = 0; thai_chars = 0\\n            with archive.open(name) as source:\\n                for row in csv.DictReader(io.TextIOWrapper(source, encoding='utf-8-sig')):\\n                    rating = int(row['star_rating']); label = int(row['label_3'])\\n                    expected = 0 if rating <= 2 else 1 if rating == 3 else 2\\n                    mismatches += expected != label\\n                    rating_counts[rating] += 1; n += 1\\n                    if split == 'test': test_truth.append(label)\\n                    thai_chars += sum('\\\\u0e00' <= c <= '\\\\u0e7f' for c in row.get('review_body', ''))\\n                    for field in counters:\\n                        key = text_key(row.get(field, ''))\\n                        if key is None: blanks[field] += 1\\n                        else:\\n                            counters[field][key] += 1\\n                            labelsets[field][key].add(label)\\n            result['splits'][split] = {'rows': n, 'rating_counts': dict(rating_counts),\\n                'label_mapping_mismatches': mismatches, 'thai_characters_in_original_column': thai_chars,\\n                'blank_text_rows': dict(blanks), 'duplicates': {\\n                    k: {'unique_texts': len(v), 'extra_repeated_rows': sum(v.values())-len(v),\\n                        'conflicting_label_groups': sum(len(s)>1 for s in labelsets[k].values())}\\n                    for k,v in counters.items()}}\\n            keys[split] = counters\\n        result['train_test_overlap'] = {}\\n        for field in keys['train']:\\n            overlap = keys['train'][field].keys() & keys['test'][field].keys()\\n            result['train_test_overlap'][field] = {'shared_unique_texts': len(overlap),\\n                'affected_train_rows': sum(keys['train'][field][k] for k in overlap),\\n                'affected_test_rows': sum(keys['test'][field][k] for k in overlap)}\\n        name = next(n for n in names if n.endswith('/outputs/best_model_preds.npz'))\\n        with zipfile.ZipFile(io.BytesIO(archive.read(name))) as arrays:\\n            true = numeric_npy(arrays.read('y_true.npy'))\\n            pred = numeric_npy(arrays.read('y_pred.npy'))\\n        if len(true) != len(pred) or true != test_truth:\\n            raise ValueError('Prediction rows do not match supplied test labels in order')\\n        if set(true + pred) - {0,1,2}:\\n            raise ValueError('Unexpected class code')\\n        matrix = [[0]*3 for _ in range(3)]\\n        for actual, predicted in zip(true, pred): matrix[actual][predicted] += 1\\n        result['prediction_order_matches_test_csv'] = True\\n        result['selected_model'] = metrics(matrix)\\n        training_counts = Counter()\\n        for rating,count in result['splits']['train']['rating_counts'].items():\\n            training_counts[0 if int(rating)<=2 else 1 if int(rating)==3 else 2] += count\\n        majority = training_counts.most_common(1)[0][0]\\n        majority_matrix = [[0]*3 for _ in range(3)]\\n        for actual in test_truth: majority_matrix[actual][majority] += 1\\n        result['training_majority_class'] = majority\\n        result['majority_baseline'] = metrics(majority_matrix)\\n        bundle_name = next(n for n in names if n.endswith('/outputs/reviewlab_bundle.json'))\\n        bundle = json.loads(archive.read(bundle_name))\\n        result['export_checks'] = {key: abs(result['selected_model'][key]-bundle['best_model']['best_metrics_3class'][key]) < 0.000051 for key in ['accuracy','macro_f1']}\\n        if not all(result['export_checks'].values()): raise ValueError('Export and prediction metrics differ')\\n    result['limitations'] = ['Training code, source release, and translation provenance absent',\\n        'Text overlap identifies evaluation risk; it does not prove how features or thresholds were fitted',\\n        'Existing test results were previously inspected; a rerun is exploratory, not a new untouched evaluation']\\n    return result\\n\\n\\nif __name__ == '__main__':\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument('--archive', default='Latest_Update.zip')\\n    parser.add_argument('--output', default='experiments/outputs/data_audit.json')\\n    args = parser.parse_args()\\n    result = audit(args.archive)\\n    destination = Path(args.output); destination.parent.mkdir(parents=True, exist_ok=True)\\n    destination.write_text(json.dumps(result, indent=2), encoding='utf-8')\\n    print(json.dumps(result, indent=2))\\n\",\"run_hybrid.py\":\"\\\"\\\"\\\"Fresh, exploratory LR versus TF-IDF + K-Means distance experiment.\\n\\nUses supplied English-cleaned CSVs, not the archive's executable model files.\\nThis is a new documented implementation, not a reproduction of missing code.\\n\\\"\\\"\\\"\\nimport argparse\\nimport csv\\nimport hashlib\\nimport io\\nimport json\\nimport platform\\nimport warnings\\nimport zipfile\\nfrom collections import Counter\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\n\\nfrom audit_project import text_key\\n\\n\\ndef load_rows(stream):\\n    rows=[]\\n    for index,row in enumerate(csv.DictReader(stream)):\\n        if not {'review_body','review_clean_en','star_rating'} <= row.keys():\\n            raise ValueError('CSV requires review_body, review_clean_en, star_rating columns')\\n        rating=int(row['star_rating'])\\n        if rating not in range(1,6): raise ValueError('Rating must be 1..5')\\n        label=0 if rating<=2 else 1 if rating==3 else 2\\n        if 'label_3' in row and int(row['label_3']) != label:\\n            raise ValueError('Rating and label_3 disagree')\\n        rows.append({'index':index,'text':row['review_clean_en'], 'key':text_key(row['review_clean_en']),\\n                     'raw_key':text_key(row['review_body']), 'label':label})\\n    return rows\\n\\n\\ndef load_csv(path):\\n    with Path(path).open(encoding='utf-8-sig',newline='') as source: return load_rows(source)\\n\\n\\ndef load_archive(path,split):\\n    with zipfile.ZipFile(path) as archive:\\n        name=next(n for n in archive.namelist() if n.endswith('/data/'+split+'_clean_en.csv'))\\n        with archive.open(name) as source: return load_rows(io.TextIOWrapper(source,encoding='utf-8-sig'))\\n\\n\\ndef grouped_ids(rows):\\n    # Union both original-text matches and cleaned-English matches.\\n    parents=list(range(len(rows)))\\n    def find(i):\\n        while parents[i]!=i:\\n            parents[i]=parents[parents[i]];i=parents[i]\\n        return i\\n    seen={}\\n    for i,row in enumerate(rows):\\n        for field in ['key','raw_key']:\\n            key=(field,row[field])\\n            if row[field] is None: continue\\n            if key in seen: parents[find(i)]=find(seen[key])\\n            else: seen[key]=i\\n    return [find(i) for i in range(len(rows))]\\n\\n\\ndef main():\\n    parser=argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument('--archive',default='Latest_Update.zip')\\n    parser.add_argument('--train-csv');parser.add_argument('--test-csv')\\n    parser.add_argument('--source-name',default='Wongnai archive (release unconfirmed)')\\n    parser.add_argument('--target-csv');parser.add_argument('--target-name')\\n    parser.add_argument('--clusters',type=int,default=16,help='Prespecified K; do not select using test results')\\n    parser.add_argument('--seed',type=int,default=42)\\n    parser.add_argument('--output',default='experiments/outputs/hybrid_rerun')\\n    args=parser.parse_args()\\n    if bool(args.train_csv)!=bool(args.test_csv):parser.error('Provide both --train-csv and --test-csv')\\n    if bool(args.target_csv)!=bool(args.target_name):parser.error('Target CSV requires a confirmed --target-name')\\n    if args.clusters<2:parser.error('--clusters must be >=2')\\n    try:\\n        import numpy as np\\n        import scipy\\n        import sklearn\\n        from scipy import sparse\\n        from sklearn.cluster import KMeans\\n        from sklearn.exceptions import ConvergenceWarning\\n        from sklearn.feature_extraction.text import TfidfVectorizer\\n        from sklearn.linear_model import LogisticRegression\\n        from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score\\n        from sklearn.model_selection import StratifiedGroupKFold\\n        from sklearn.preprocessing import StandardScaler\\n    except ImportError as exc:\\n        raise SystemExit('Install requirements first: python -m pip install -r experiments/requirements.txt') from exc\\n    # A failed convergence check invalidates the trial rather than silently publishing it.\\n    warnings.simplefilter('error',ConvergenceWarning)\\n    train=load_csv(args.train_csv) if args.train_csv else load_archive(args.archive,'train')\\n    test=load_csv(args.test_csv) if args.test_csv else load_archive(args.archive,'test')\\n    target=load_csv(args.target_csv) if args.target_csv else []\\n    output=Path(args.output)\\n    if (output/'experiment.json').exists():\\n        raise SystemExit('Output already contains a run. Use --output with a new directory to preserve it.')\\n    output.mkdir(parents=True,exist_ok=True)\\n    removed=Counter()\\n    evaluation_rows=test+target\\n    blocked={field:{r[field] for r in evaluation_rows if r[field]} for field in ['key','raw_key']}\\n    retained=[];seen=set()\\n    for row in train:\\n        if not row['key']:removed['blank_training_text']+=1;continue\\n        if any(row[field] and row[field] in blocked[field] for field in blocked):\\n            removed['training_overlap_with_evaluation_text']+=1;continue\\n        pair=(row['key'],row['label'])\\n        if pair in seen:removed['repeated_clean_text_and_label']+=1;continue\\n        seen.add(pair);retained.append(row)\\n    train=retained\\n    # Keep the complete test set (including empty texts) so test support is explicit.\\n    groups=np.asarray(grouped_ids(train))\\n    y=np.asarray([r['label'] for r in train])\\n    text=np.asarray([r['text'] for r in train],dtype=object)\\n    if set(y)!={0,1,2} or min(Counter(y).values())<5:\\n        raise ValueError('Need all three classes with at least five training examples each')\\n    splitter=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=args.seed)\\n    fit_idx,val_idx=next(splitter.split(text,y,groups))\\n    if set(y[fit_idx])!={0,1,2} or set(y[val_idx])!={0,1,2}:\\n        raise ValueError('Validation split lacks a class; revise the prespecified grouping protocol')\\n    assert not(set(groups[fit_idx]) & set(groups[val_idx]))\\n    vectorizer_args={'ngram_range':(1,2),'min_df':2,'max_features':50000,'sublinear_tf':True,'dtype':np.float64}\\n    def vectorizer():return TfidfVectorizer(**vectorizer_args)\\n    def clusterer():return KMeans(n_clusters=args.clusters,n_init=10,max_iter=300,random_state=args.seed,algorithm='lloyd')\\n    def classifier(c):return LogisticRegression(C=c,class_weight='balanced',solver='lbfgs',max_iter=3000,random_state=args.seed)\\n    def combined(x,km,scaler,fit=False):\\n        distances=km.transform(x)\\n        distances=scaler.fit_transform(distances) if fit else scaler.transform(distances)\\n        return sparse.hstack([x,sparse.csr_matrix(distances)],format='csr')\\n    vec=vectorizer();x_fit=vec.fit_transform(text[fit_idx]);x_val=vec.transform(text[val_idx])\\n    if args.clusters>=len(fit_idx):raise ValueError('K exceeds training-fold sample count')\\n    km=clusterer();km.fit(x_fit)\\n    scaler=StandardScaler();h_fit=combined(x_fit,km,scaler,True);h_val=combined(x_val,km,scaler)\\n    trials=[];best={}\\n    # Equal model-selection budget: the same three C values per arm; K is fixed in advance.\\n    for arm,a,b in [('ordinary_lr',x_fit,x_val),('kmeans_distance_lr',h_fit,h_val)]:\\n        candidates=[]\\n        for c in [0.1,1.0,10.0]:\\n            model=classifier(c);model.fit(a,y[fit_idx]);pred=model.predict(b)\\n            score=float(f1_score(y[val_idx],pred,labels=[0,1,2],average='macro',zero_division=0))\\n            trials.append({'arm':arm,'C':c,'validation_macro_f1':score})\\n            candidates.append((score,-c,c))\\n        best[arm]=max(candidates)[2]\\n    # Refit all learned components on development data after model selection; no test fitting.\\n    vec=vectorizer();x_train=vec.fit_transform(text)\\n    km=clusterer();km.fit(x_train);scaler=StandardScaler();h_train=combined(x_train,km,scaler,True)\\n    models={}\\n    for arm,x in [('ordinary_lr',x_train),('kmeans_distance_lr',h_train)]:\\n        model=classifier(best[arm]);model.fit(x,y);models[arm]=model\\n    majority=int(Counter(y).most_common(1)[0][0])\\n    results=[]\\n    for target_name,rows in [(args.source_name,test)]+([(args.target_name,target)] if target else []):\\n        truth=np.asarray([r['label'] for r in rows]);x=vec.transform([r['text'] for r in rows]);h=combined(x,km,scaler)\\n        for arm,features in [('ordinary_lr',x),('kmeans_distance_lr',h),('training_majority',None)]:\\n            predictions=np.full(len(rows),majority) if features is None else models[arm].predict(features)\\n            results.append({'source':args.source_name,'target':target_name,'arm':arm,'support':len(rows),\\n                'accuracy':float(accuracy_score(truth,predictions)),\\n                'macro_f1':float(f1_score(truth,predictions,labels=[0,1,2],average='macro',zero_division=0)),\\n                'per_class':classification_report(truth,predictions,labels=[0,1,2],target_names=['Negative','Neutral','Positive'],output_dict=True,zero_division=0),\\n                'confusion_matrix':confusion_matrix(truth,predictions,labels=[0,1,2]).tolist()})\\n        filename='within_platform_predictions.npz' if rows is test else 'transfer_predictions.npz'\\n        np.savez_compressed(output/filename,y_true=truth,ordinary_lr=models['ordinary_lr'].predict(x),hybrid_lr=models['kmeans_distance_lr'].predict(h))\\n    source_results=[r for r in results if r['target']==args.source_name]\\n    delta=next(r['macro_f1'] for r in source_results if r['arm']=='kmeans_distance_lr')-next(r['macro_f1'] for r in source_results if r['arm']=='ordinary_lr')\\n    report={'created_utc':datetime.now(timezone.utc).isoformat(),'new_implementation':True,\\n        'status':'Exploratory rerun; original test results were already inspected',\\n        'seed':args.seed,'clusters_prespecified':args.clusters,'C_candidates':[0.1,1.0,10.0],\\n        'selected_C':best,'class_weight':'balanced','tfidf':{k:str(v) for k,v in vectorizer_args.items()},\\n        'split':'First fold of 5-fold StratifiedGroupKFold used as fixed validation; NOT averaged cross-validation',\\n        'development_rows':len(train),'fit_rows':len(fit_idx),'validation_rows':len(val_idx),'removed_training_rows':dict(removed),\\n        'training_row_indices':[r['index'] for r in train],'fit_indices':fit_idx.tolist(),'validation_indices':val_idx.tolist(),\\n        'versions':{'python':platform.python_version(),'numpy':np.__version__,'scipy':scipy.__version__,'sklearn':sklearn.__version__},\\n        'validation_trials':trials,'results':results,'hybrid_minus_ordinary_macro_f1':delta,\\n        'limitations':['Translation provenance and source release require external confirmation',\\n            'Fixed historical test set is exploratory, not newly untouched',\\n            'No claim of reproducing the old missing training implementation',\\n            'Distance features are scaled; TF-IDF features retain their L2 normalization',\\n            'No statistical significance or guaranteed hybrid improvement claimed']}\\n    if not args.train_csv:\\n        with Path(args.archive).open('rb') as source:report['archive_sha256']=hashlib.file_digest(source,'sha256').hexdigest()\\n    else:\\n        report['source_csv_sha256']={}\\n        for filename in [args.train_csv,args.test_csv]:\\n            with Path(filename).open('rb') as source:report['source_csv_sha256'][filename]=hashlib.file_digest(source,'sha256').hexdigest()\\n    if args.target_csv:\\n        with Path(args.target_csv).open('rb') as source:report['target_csv_sha256']=hashlib.file_digest(source,'sha256').hexdigest()\\n    (output/'experiment.json').write_text(json.dumps(report,indent=2),encoding='utf-8')\\n    with (output/'metrics.csv').open('w',newline='',encoding='utf-8') as stream:\\n        writer=csv.DictWriter(stream,fieldnames=['source','target','arm','support','accuracy','macro_f1'],extrasaction='ignore')\\n        writer.writeheader();writer.writerows(results)\\n    print(json.dumps({'output':str(output),'hybrid_minus_ordinary_macro_f1':delta,'results':[{k:r[k] for k in ['source','target','arm','accuracy','macro_f1']} for r in results]},indent=2))\\n\\n\\nif __name__=='__main__':main()\\n\",\"requirements.txt\":\"numpy>=1.26,<3\\nscipy>=1.11,<2\\nscikit-learn>=1.4,<2\\n\"}")
for name, content in source_files.items():
    Path(name).write_text(content, encoding="utf-8")
print("Experiment scripts prepared.")

## Upload your archive

Choose `Latest_Update.zip` from your computer. Only upload data you are authorized to process in Colab. The scripts read CSV and numeric arrays and do not load the saved joblib models.

In [ ]:
from google.colab import files
uploaded = files.upload()
archive_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if archive_name is None:
    raise ValueError("Upload Latest_Update.zip")
if archive_name != "Latest_Update.zip":
    Path(archive_name).replace("Latest_Update.zip")
del uploaded
print("Archive ready.")

In [ ]:
%run audit_project.py --archive Latest_Update.zip --output colab_audit.json

## Run the paired experiment

Both arms use balanced weights and the same three C candidates. K=16 is fixed in advance. Validation groups keep matching source/translated texts together. Source training texts that overlap evaluation texts are excluded. Training-only TF-IDF, K-Means and distance scaling are refitted after validation selection. This may take several minutes.

In [ ]:
%run run_hybrid.py --archive Latest_Update.zip --output colab_results

In [ ]:
import json
report = json.loads(Path("colab_results/experiment.json").read_text())
for run in report["results"]:
    print(run["arm"], "Accuracy:", round(run["accuracy"], 4), "Macro-F1:", round(run["macro_f1"], 4))
print("Hybrid minus ordinary Macro-F1:", report["hybrid_minus_ordinary_macro_f1"])

In [ ]:
import zipfile
with zipfile.ZipFile("trendminers_experiment_outputs.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write("colab_audit.json")
    for item in Path("colab_results").iterdir():
        if item.is_file():
            archive.write(item, "colab_results/" + item.name)
files.download("trendminers_experiment_outputs.zip")

Upload the downloaded results ZIP to this conversation for website integration. Genuine Google Maps records with rating-based ground truth are still required for cross-platform evaluation; the unlabeled review predictions are not ground truth.